In [65]:
import torch
from torch import nn

from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
import random
import numpy as np

from typing import Tuple, List, Optional
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime



import lightning as pl


from Models.models import Siren, Finer
from Utils.utils import get_full_img, norm, get_device, dice_stack_helper, get_model, ClearCache
from Data.load_data_3d import load_data, get_gt_seg
from Utils.defaults import default_config
from Utils.plotting_utils2 import plot_seg_results_paper, plot_final_results_paper, plot_hf_results_paper
from Utils.plotting_utils import loss_plot, plot_image_metrics, plot_4_images
from LFSynth.ContrastModulation import ContrastModulation
from test3D import visualize_volume_slices
import copy

In [88]:
config = copy.deepcopy(default_config)
config["in_features"] = 3
hf_ground_truth, lf_gt, prior_seg_dice, lf_gt_seg_dice, M = load_data(1, config) #uncomment
gt_image = torch.tensor(norm(hf_ground_truth)).unsqueeze(-1)
gt_image = gt_image.to(torch.float32)

lf_gt = torch.tensor(norm(lf_gt)).unsqueeze(-1)
lf_gt = lf_gt.to(torch.float32)
print('gt_image loaded')

Device =  mps
torch.Size([87, 96, 192])
BG Noise in different regions : 4.7445857300717 4.415978248235479 4.956849146645869 4.8247938816077935
known_m =  [0.75 0.9  0.9 ]
[[ 28.84998854   0.          -7.89492106]
 [ 28.84998854 -22.35619149   0.        ]
 [  0.          22.35619149  -7.89492106]] [14.53206245  1.51691906 13.01514339]
             m_init  epsilon  \
0   [0.1, 0.1, 0.1]     0.00   
1   [0.1, 0.1, 0.1]     0.01   
2   [0.1, 0.1, 0.1]     0.05   
3   [0.1, 0.1, 0.1]     0.10   
4   [0.1, 0.1, 0.1]     0.20   
..              ...      ...   
59  [0.5, 0.5, 0.5]     0.10   
60  [0.5, 0.5, 0.5]     0.20   
61  [0.5, 0.5, 0.5]     0.30   
62  [0.5, 0.5, 0.5]     0.40   
63  [0.5, 0.5, 0.5]     0.50   

                                                 loss  \
0   [144.631340676391, 144.63133495698293, 144.631...   
1   [144.63164067639102, 144.63163495701272, 144.6...   
2   [144.632840676391, 144.63283495713193, 144.632...   
3   [144.634340676391, 144.63433495728094, 144.634.

In [4]:
type(lf_gt), lf_gt.shape, hf_ground_truth.shape, type(hf_ground_truth)

(numpy.ndarray, (87, 96, 192), (174, 192, 192), numpy.ndarray)

In [100]:
class RandomPointsDataset(Dataset):
    def __init__(self, image: torch.Tensor, lf_image:torch.Tensor, points_num: int = POINTS_PER_SAMPLE):
        super().__init__()
        self.device = get_device()
        self.points_num = points_num
        assert image.dtype == torch.float32
        assert lf_image.dtype == torch.float32
        self.image = image.to(self.device)  # (H, W, ..., C)
        self.lf_image = lf_image.to(self.device)  # (H, W, ..., C)
        self.dim_sizes = self.image.shape[:-1]  # Size of each spatial dimension

        # To help us define the input/output sizes of our network later
        # we store the size of our input coordinates and output values
        self.coord_size = len(self.image.shape[:-1])  # Number of spatial dimensions
        self.value_size = self.image.shape[-1]  # Channel size

    def __len__(self):
        return 1

    def __getitem__(self, idx: int):
        # Create random sample of pixel indices
        point_indices = [torch.randint(0, i, (self.points_num,), device=self.device) for i in self.dim_sizes]
        point_indices = [i.to(torch.int32) for i in point_indices]
        print(point_indices[0].dtype)
        point_indices_lf = [(F.interpolate(i.unsqueeze(0).unsqueeze(0), scale_factor=0.25)).squeeze(0).squeeze(0) for i in point_indices]
        # Retrieve image values from selected indices
        point_values = self.image[tuple(point_indices)]
        point_values_lf = self.lf_image[tuple(point_indices_lf)]
        print(point_values.shape, point_values_lf.shape)
        
        # Convert point indices into normalized [-1.0, 1.0] coordinates
        point_coords = torch.stack(point_indices, dim=-1)
        spatial_dims = torch.tensor(self.dim_sizes, device=self.device)
        point_coords_norm = point_coords / (spatial_dims / 2) - 1

        # The subject index is also returned in case the user wants to use subject-wise learned latents
        return point_coords_norm, point_values, point_values_lf

In [101]:
POINTS_PER_SAMPLE = 96*96*4
dataset = RandomPointsDataset(gt_image, lf_gt, points_num=POINTS_PER_SAMPLE)
dataloader = DataLoader(dataset, batch_size=1, num_workers=0, pin_memory=False) # We set a batch_size of 1 since our dataloader is already returning a batch of points.

Device =  mps


In [102]:
temp = next(iter(dataloader))

torch.int32
torch.Size([36864, 1]) torch.Size([9216, 1])
